# MIMIC-IV Sepsis Data Preprocessing

This notebook preprocesses MIMIC-IV data for the Antigravity multi-agent sepsis prediction system.

## Processing Pipeline:
1. Load MIMIC-IV data from Google Drive
2. Harmonize to CinC 2019 schema (40 canonical variables)
3. Calculate SOFA scores
4. Generate Sepsis-3 labels
5. Save processed data to HDF5 format

## Modes:
- **test**: 100 patients (~1 hour, 50MB output)
- **medium**: 5,000 patients (~12 hours, 2GB output)
- **full**: All ICU patients (~24-48 hours, 30GB+ output)

**Author**: Jason  
**Date**: January 2026  
**Deadline**: February 5, 2026 (Supervisor Meeting)

## 1. Setup & Configuration

In [ ]:
# ========================================
# MODE SELECTION (Change this line to scale up)
# ========================================
MODE = "test"  # Options: "test", "medium", "full"

MODE_CONFIG = {
    "test": {"n_patients": 100, "description": "Quick test (1 hour)"},
    "medium": {"n_patients": 5000, "description": "Medium scale (12 hours)"},
    "full": {"n_patients": None, "description": "Full dataset (24-48 hours)"}
}

print(f"🚀 Running in {MODE.upper()} mode: {MODE_CONFIG[MODE]['description']}")
print(f"   Patients to process: {MODE_CONFIG[MODE]['n_patients'] or 'ALL'}")

In [ ]:
# Install dependencies
!pip install -q pandas numpy pyyaml h5py tqdm scikit-learn

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Set up paths
import os

# Update these paths to match your Google Drive structure
DRIVE_ROOT = "/content/drive/MyDrive"
MIMIC_RAW_PATH = f"{DRIVE_ROOT}/MIMIC"  # Where your MIMIC data is stored
PROJECT_PATH = f"{DRIVE_ROOT}/Sepsis"   # Your project folder
OUTPUT_PATH = f"{PROJECT_PATH}/data/processed/mimic_harmonized"

# Create output directory
os.makedirs(OUTPUT_PATH, exist_ok=True)

print(f"📁 MIMIC data location: {MIMIC_RAW_PATH}")
print(f"📁 Output location: {OUTPUT_PATH}")

In [ ]:
# Copy project code to Colab environment
import sys
sys.path.append(f"{PROJECT_PATH}/src")

# Verify code is accessible
!ls -R "{PROJECT_PATH}/src/data"

In [ ]:
# Import preprocessing modules
import pandas as pd
import numpy as np
import yaml
import h5py
from tqdm.notebook import tqdm
import logging

from data.sofa_calculator import SOFACalculator
from data.harmonization import MIMICHarmonizer
from data.labeling import SepsisLabeler

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("✅ Modules imported successfully")

## 2. Load Configuration

In [ ]:
# Load configuration
config_path = f"{PROJECT_PATH}/config/data_config.yaml"

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

print("📋 Configuration loaded:")
print(f"   - Variables to extract: {len(config['variable_mapping'])}")
print(f"   - Prediction window: {config['sepsis_definition']['prediction_window']}")
print(f"   - SOFA baseline strategy: {config['sepsis_definition']['sofa']['baseline_calculation']}")

## 3. Load MIMIC-IV Data (Chunked)

We load data in chunks to avoid memory errors.

In [ ]:
# Load patient cohort (ICU stays)
print("📥 Loading patient cohort...")
icustays = pd.read_csv(f"{MIMIC_RAW_PATH}/icustays.csv.gz")
patients = pd.read_csv(f"{MIMIC_RAW_PATH}/patients.csv.gz")
admissions = pd.read_csv(f"{MIMIC_RAW_PATH}/admissions.csv.gz")

print(f"   Total ICU stays: {len(icustays)}")

# Select patient subset BEFORE merging (more memory efficient)
n_patients = MODE_CONFIG[MODE]['n_patients']
if n_patients:
    icustays = icustays.head(n_patients)
    print(f"   Selected {n_patients} ICU stays for {MODE} mode")

# Merge to get demographics
cohort = icustays.merge(patients, on='subject_id').merge(admissions, on=['subject_id', 'hadm_id'])

# Convert timestamps
cohort['intime'] = pd.to_datetime(cohort['intime'])
cohort['outtime'] = pd.to_datetime(cohort['outtime'])

print(f"\n✅ Cohort ready: {len(cohort)} patients to process")

# Clean up to free memory
del icustays, patients, admissions
import gc
gc.collect()

In [ ]:
# Load item dictionaries (for decoding itemids)
print("📥 Loading item dictionaries...")
d_items = pd.read_csv(f"{MIMIC_RAW_PATH}/d_items.csv.gz")
d_labitems = pd.read_csv(f"{MIMIC_RAW_PATH}/d_labitems.csv.gz")

print(f"   Chart items: {len(d_items)}")
print(f"   Lab items: {len(d_labitems)}")

## 4. Initialize Preprocessing Modules

In [ ]:
# Initialize harmonizer
harmonizer = MIMICHarmonizer(config_path)

# Initialize labeler
labeler = SepsisLabeler(config['sepsis_definition'])

print("✅ Preprocessing modules initialized")

In [ ]:
# Memory optimization: Clean up item dictionaries (no longer needed)
import gc
del d_items, d_labitems
gc.collect()

print("🧹 Memory cleaned up")

In [ ]:
# Load chartevents and labevents for ALL patients in cohort at once
# This is MUCH faster than loading per patient

chartevents_file = f"{MIMIC_RAW_PATH}/chartevents.csv.gz"
labevents_file = f"{MIMIC_RAW_PATH}/labevents.csv.gz"

# Get cohort subject IDs
cohort_subject_ids = set(cohort['subject_id'].unique())

print(f"📥 Loading chartevents for {len(cohort_subject_ids)} patients...")
chartevents_chunks = []
for chunk in pd.read_csv(chartevents_file, chunksize=100000, low_memory=False):
    # Filter to cohort patients only
    filtered = chunk[chunk['subject_id'].isin(cohort_subject_ids)]
    if len(filtered) > 0:
        chartevents_chunks.append(filtered)

chartevents_all = pd.concat(chartevents_chunks) if chartevents_chunks else pd.DataFrame()
# Convert charttime to datetime
chartevents_all['charttime'] = pd.to_datetime(chartevents_all['charttime'])
print(f"   Chartevents loaded: {len(chartevents_all)} records")

print(f"📥 Loading labevents for {len(cohort_subject_ids)} patients...")
labevents_chunks = []
for chunk in pd.read_csv(labevents_file, chunksize=100000, low_memory=False):
    # Filter to cohort patients only
    filtered = chunk[chunk['subject_id'].isin(cohort_subject_ids)]
    if len(filtered) > 0:
        labevents_chunks.append(filtered)

labevents_all = pd.concat(labevents_chunks) if labevents_chunks else pd.DataFrame()
# Convert charttime to datetime
labevents_all['charttime'] = pd.to_datetime(labevents_all['charttime'])
print(f"   Labevents loaded: {len(labevents_all)} records")

print(f"\n✅ All clinical data loaded for cohort")

In [ ]:
# Load prescriptions and microbiology for cohort patients
prescriptions_file = f"{MIMIC_RAW_PATH}/prescriptions.csv.gz"
microbiology_file = f"{MIMIC_RAW_PATH}/microbiologyevents.csv.gz"

print(f"📥 Loading prescriptions (filtered by cohort)...")
prescriptions_chunks = []
for chunk in pd.read_csv(prescriptions_file, chunksize=50000, low_memory=False):
    filtered_chunk = chunk[chunk['subject_id'].isin(cohort_subject_ids)]
    if len(filtered_chunk) > 0:
        prescriptions_chunks.append(filtered_chunk)

prescriptions = pd.concat(prescriptions_chunks) if prescriptions_chunks else pd.DataFrame()
prescriptions['starttime'] = pd.to_datetime(prescriptions['starttime'])
print(f"   Prescriptions loaded: {len(prescriptions)} records")

print(f"📥 Loading microbiology (filtered by cohort)...")
microbiology_chunks = []
for chunk in pd.read_csv(microbiology_file, chunksize=50000, low_memory=False):
    filtered_chunk = chunk[chunk['subject_id'].isin(cohort_subject_ids)]
    if len(filtered_chunk) > 0:
        microbiology_chunks.append(filtered_chunk)

microbiology = pd.concat(microbiology_chunks) if microbiology_chunks else pd.DataFrame()
microbiology['charttime'] = pd.to_datetime(microbiology['charttime'])
print(f"   Microbiology loaded: {len(microbiology)} records")

print(f"\n✅ All data loaded for cohort")

In [ ]:
# Process all patients
processed_patients = []
sepsis_count = 0

print(f"\n🔄 Processing {len(cohort)} patients...\n")

for idx, row in tqdm(cohort.iterrows(), total=len(cohort), desc="Processing patients"):
    try:
        subject_id = row['subject_id']
        hadm_id = row['hadm_id']
        icu_intime = row['intime']
        icu_outtime = row['outtime']
        
        # Filter data for this patient from pre-loaded data
        patient_chartevents = chartevents_all[chartevents_all['subject_id'] == subject_id]
        patient_labevents = labevents_all[labevents_all['subject_id'] == subject_id]
        
        if len(patient_chartevents) == 0 and len(patient_labevents) == 0:
            continue
        
        # Harmonize to CinC schema
        harmonized = harmonizer.harmonize_patient(
            subject_id, patient_chartevents, patient_labevents, icu_intime, icu_outtime
        )
        
        if len(harmonized) == 0:
            continue
        
        # Get patient prescriptions and cultures
        patient_prescriptions = prescriptions[prescriptions['subject_id'] == subject_id]
        patient_microbiology = microbiology[microbiology['subject_id'] == subject_id]
        
        # Generate Sepsis-3 labels
        labeled, has_sepsis = labeler.label_patient(
            subject_id, harmonized, patient_prescriptions, patient_microbiology,
            icu_intime, icu_outtime
        )
        
        if has_sepsis:
            sepsis_count += 1
        
        # Add metadata
        labeled['hadm_id'] = hadm_id
        labeled['icu_intime'] = icu_intime
        labeled['icu_outtime'] = icu_outtime
        
        processed_patients.append(labeled)
        
    except Exception as e:
        logger.error(f"Error processing patient {subject_id}: {e}")
        continue

print(f"\n✅ Processing complete!")
print(f"   Processed: {len(processed_patients)} patients")
print(f"   Sepsis cases: {sepsis_count}")
if len(processed_patients) > 0:
    print(f"   Sepsis prevalence: {sepsis_count/len(processed_patients)*100:.1f}%")
else:
    print(f"   No patients were successfully processed")

## 6. Combine and Save Results

In [ ]:
# Combine all patients
if len(processed_patients) > 0:
    all_data = pd.concat(processed_patients, ignore_index=True)
    print(f"📊 Total observations: {len(all_data)}")
    print(f"   Positive labels: {(all_data['sepsis_label']==1).sum()}")
    print(f"   Negative labels: {(all_data['sepsis_label']==0).sum()}")
else:
    print("❌ No patients processed successfully")

In [ ]:
# Save to HDF5 format (efficient for large datasets)
output_file = f"{OUTPUT_PATH}/mimic_processed_{MODE}.h5"

print(f"💾 Saving to {output_file}...")
all_data.to_hdf(output_file, key='data', mode='w', complevel=9)

print(f"✅ Data saved successfully!")
print(f"   File size: {os.path.getsize(output_file) / 1024 / 1024:.1f} MB")

In [ ]:
# Save summary statistics
summary = {
    'mode': MODE,
    'n_patients': len(processed_patients),
    'n_observations': len(all_data),
    'n_sepsis_cases': sepsis_count,
    'sepsis_prevalence': sepsis_count / len(processed_patients),
    'positive_labels': (all_data['sepsis_label']==1).sum(),
    'negative_labels': (all_data['sepsis_label']==0).sum(),
    'variables': list(all_data.columns)
}

summary_file = f"{OUTPUT_PATH}/summary_{MODE}.json"
import json
with open(summary_file, 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print(f"📄 Summary saved to {summary_file}")

## 7. Validation & Quality Checks

In [ ]:
# Check data quality
print("🔍 Data Quality Checks:\n")

# Missing value rates
print("Missing Value Rates:")
missing_rates = all_data.isnull().mean().sort_values(ascending=False)
print(missing_rates[missing_rates > 0].head(10))

print("\nLabel Distribution:")
print(all_data['sepsis_label'].value_counts())

print("\nSample Statistics:")
numeric_cols = all_data.select_dtypes(include=[np.number]).columns[:10]
print(all_data[numeric_cols].describe())

In [ ]:
# Visualize label distribution
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Label counts
all_data['sepsis_label'].value_counts().plot(kind='bar', ax=axes[0])
axes[0].set_title('Label Distribution')
axes[0].set_xlabel('Label')
axes[0].set_ylabel('Count')

# Time to onset distribution
positive_data = all_data[all_data['sepsis_label'] == 1]
if 'hours_to_onset' in positive_data.columns:
    positive_data['hours_to_onset'].hist(bins=50, ax=axes[1])
    axes[1].set_title('Hours to Sepsis Onset (Positive Labels)')
    axes[1].set_xlabel('Hours')
    axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig(f"{OUTPUT_PATH}/label_distribution_{MODE}.png", dpi=150)
plt.show()

print(f"📊 Plots saved to {OUTPUT_PATH}/label_distribution_{MODE}.png")

## 8. Next Steps

### ✅ What you now have:
- Processed MIMIC-IV data in CinC schema
- SOFA scores calculated
- Sepsis-3 labels generated
- Data saved in HDF5 format

### 🔜 What to do next:
1. **For supervisor meeting (Feb 5)**: Show this notebook running in test mode
2. **After approval**: Run in `"medium"` or `"full"` mode (change MODE at top)
3. **Model training**: Use processed data to train multi-agent system
4. **External validation**: Process CinC 2019 data similarly

### 📈 To scale up:
```python
# Change MODE at the top of notebook:
MODE = "medium"  # or "full"
```

### 💡 Tips:
- **Free Colab**: Stick with "test" mode
- **Colab Pro ($10/month)**: Can run "medium" mode
- **Cloud VM**: Run "full" mode overnight